# 02c - Time masks of the stored features

For the network with time masking (`model.time_masking: true`): marks, for every record of the feature file used in
training, which of the time bins of its window contain recorded data. A bin is data when its centre lies between the
first sample recorded by any component and the last sample recorded by any component; outside, the window holds only
the zero padding added to align the traces. Missing data of a single component and gaps inside the recording are not
masked: they are defects that the network has to see.

- Feature file: `training.feature_source` with the window placement of `preprocessing.window.placement`.
- Output: `time_masks.h5` next to the feature file, one row per record in the order of `feature_index`.
- Only the traces are read (no transforms): a few tens of minutes on the full dataset. If interrupted, run it again
  (it restarts from the beginning; no incomplete file is left).

## 0. Setup

In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np

from wqu.config import load_config
from wqu.data.waveforms import iter_waveform_batches
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    check_feature_placement,
    feature_file_path,
    grid_from_config,
    read_feature_records,
)
from wqu.preprocessing.time_masks import compute_time_masks, mask_summary, read_time_masks, time_mask_path

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
SEISBENCH_SOURCE = "seisbench"

config = load_config(CONFIG_PATH)
feature_source = config.training.feature_source
if feature_source == SEISBENCH_SOURCE:
    # Traces from the SeisBench datasets, whatever data.source and waveforms.source say.
    config = replace(
        config,
        data=replace(config.data, source=SEISBENCH_SOURCE),
        waveforms=replace(config.waveforms, source=SEISBENCH_SOURCE),
    )
columns = config.columns
preprocessing = config.preprocessing
label_names = {subset.label: subset.name for subset in config.data.subsets}
features_path = feature_file_path(preprocessing, feature_source)
check_feature_placement(features_path, preprocessing)
stored_records = read_feature_records(features_path, columns.text)
n_times = len(grid_from_config(preprocessing).times_s)
masks_path = time_mask_path(features_path)
print(f"Feature file: {features_path} ({len(stored_records):,} records, {preprocessing.window_placement} windows, "
      f"{n_times} time bins)")
print(f"Output: {masks_path}")

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature file: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/full/features.h5 (100,249 records, full windows, 128 time bins)
Output: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/full/time_masks.h5


## 1. Time masks

Skipped if the mask file already exists.

In [2]:
if masks_path.exists():
    masks = read_time_masks(masks_path)
    print(f"Mask file already complete: {masks.shape[0]:,} records")
else:
    masks = compute_time_masks(stored_records, lambda selected: iter_waveform_batches(selected, config), n_times,
                               config.waveforms.zero_tolerance, masks_path)
    print(f"Stored the masks of {masks.shape[0]:,} records in {masks_path}")
if masks.shape != (len(stored_records), n_times):
    raise ValueError(f"Mask file of shape {masks.shape}, expected {(len(stored_records), n_times)}.")

time masks: 100%|██████████| 100249/100249 [01:35<00:00, 1054.22it/s]


Stored the masks of 100,249 records in /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/full/time_masks.h5


## 2. Checks

Per class: records, records without any data bin (excluded from training and evaluation with time masking, and
classified bad by rule) and mean share of data bins. Then the distribution of the share of data bins.

In [3]:
summary = mask_summary(stored_records, masks, columns.label).rename(index=label_names)
summary.round(3)

,records,without_data,data_share_mean
label,,,
good,64213,0,0.592
bad,36036,157,0.659


In [4]:
data_share = masks[stored_records[FEATURE_INDEX_COLUMN].to_numpy()].mean(axis=1)
quantiles = [0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0]
(stored_records.assign(data_share=data_share).groupby(columns.label)["data_share"].quantile(quantiles)
 .unstack().rename(index=label_names).round(3))

,0.00,0.05,0.25,0.50,0.75,0.95,1.00
label,,,,,,,
good,0.016,0.117,0.438,0.664,0.719,0.914,1.0
bad,0.000,0.211,0.648,0.688,0.742,0.914,1.0
